# Implementação da LeNet-5 em PyTorch (MNIST)

Este notebook contém o passo a passo para implementar, treinar e analisar a rede neural clássica **LeNet-5** no dataset **MNIST**.

### 1. **Definição do Modelo**: Implementação da classe LeNet5 conforme o artigo original.

In [ ]:
import torch # Importa a biblioteca principal do PyTorch
import torch.nn as nn # Para criar as camadas com parâmetros (pesos/viés) aprendíveis (Conv2d, Linear, etc.)
import torch.nn.functional as F 

class LeNet5(nn.Module):
    # Definir num_classes como argumento torna o modelo escalável e dinâmico para outras bases
    def __init__(self, num_classes=10): 
        super(LeNet5, self).__init__() 
        
        # --- CAMADAS DE CONVOLUÇÃO ---
        # Recebe 1 canal (ex: imagens MNIST em escala de cinza).
        # MUITO IMPORTANTE (PADDING): O MNIST tem imagens originais de dimensão 28x28. A LeNet5 espera entradas 32x32.
        # Ao adicionar padding=2, inserimos 2 pixels de borda (zeros) ao redor da imagem 28x28, transformando a entrada de 28x28 para 32x32.
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=2)
        
        # Poolings instanciados separadamente (pool1 e pool2) para facilitar visualização da arquitetura camada por camada.
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Recebe os 6 canais e extrai 16 canais de características.
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Após as subamostragens, as feições chegam como 5x5. Um kernel 5x5 extrai um valor 1x1.
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1)
        
        # Recebe as 120 características das convoluções e mapeia para 84 caminhos para aprendizado.
        self.fc1 = nn.Linear(in_features=120, out_features=84)
        
        # Última camada de classificação, parametrizada para aceitar N classes.
        self.fc2 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):
        # O agrupamento encadeado de funções torna o forward extremamente legível e enxuto.
        # Trocamos a tradicional Ativação Tanh pela ativação ReLU, garantindo um treinamento mais rápido e efetivo sem estagnação dos gradientes!
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        
        # x.view() é o análogo clássico do flatten. 
        # Aqui, mantemos do tamanho de batch `x.size(0)` exatamente como está, 
        # e o `-1` instrui o PyTorch a comprimir todo o resto num único vetor 1D!
        x = x.view(x.size(0), -1)
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x


### 2. **Visualização de Filtros**: Inspeção dos pesos iniciais da primeira camada de convolução.

In [ ]:
import matplotlib.pyplot as plt

def visualizar_filtros_conv1():
    # Instancia o modelo
    model = LeNet5()
    
    # Captura os pesos numéricos (filtros) da primeira camada de convolução
    # Como conv1 possui 6 filtros de extração (out_channels=6), para 1 canal de entrada, de tamanho 5x5.
    filters = model.conv1.weight.data
    
    print(f"Estrutura do Tensor dos Filtros: {filters.shape}")
    print("Formato: (Qtd. de Filtros, Profundidade/Canais, Altura, Largura)\n")
    print("Atenção aos blocos: eles começam como um 'ruído aleatório' contendo os valores que ")
    print("sua rede tentará otimizar via Backpropagation.\n")
    
    # Plota os 6 filtros lado a lado para visualizarmos a aleatoriedade inicial antes do treino
    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    
    for i in range(6):
        # Extrai o filtro 'i', no canal matricial de origem '0', e converte pra numpy
        f = filters[i, 0, :, :].numpy()
        
        # Utilizar 'viridis' ou 'gray' ajuda a destacar diferenças nos pesos positivos e negativos
        im = axes[i].imshow(f, cmap='viridis')
        axes[i].set_title(f'Filtro {i+1}')
        axes[i].axis('off')
    
    plt.suptitle("Estado Inicial Aleatório dos Kernels de Convolução 5x5", fontsize=14, y=1.05)
    plt.show()

# Chama e executa o bloco visual
visualizar_filtros_conv1()


### 3. **Carregamento de Dados**: Download e visualização de amostras do dataset MNIST.

In [ ]:
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 1. Definimos o fluxo de Transformação da Imagem
# O ToTensor() carrega as matrizes numéricas das imagens já convertendo elas para Tensores do PyTorch
# e muda o escopo de pixels numéricos de [0, 255] para formato padronizado [0.0, 1.0].
transform = transforms.Compose([
    transforms.ToTensor()
])

# 2. Baixar e alocar o dataset MNIST (Números digitados à mão)
# Se você não tem na pasta './data', o PyTorch usa o parâmetro 'download=True' para resolver tudo automático.
print("Verificando/Baixando os arquivos do MNIST...\n")
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# 3. Criar nosso iterador (DataLoader)
# O batch_size=5 pegará de 5 em 5. Shuffle=True embaralha a base para evitar que a rede 'decore' a sequência.
train_loader_visual = torch.utils.data.DataLoader(train_dataset, batch_size=5, shuffle=True)

# Iteramos e extraímos o nosso primeiro lote (batch) de imagens e de etiquetas reais (rótulos)
dataiter = iter(train_loader_visual)
imagens, rotulos = next(dataiter)

# 4. Plotagem
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    # As imagens retornam do pyTorch no formato (1, 28, 28). 
    # Precisamos remover a dimensão "fantasma" do canal usando '.squeeze()', extraindo somente os pixels 28x28 para o Matplotlib
    img = imagens[i].squeeze()
    
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Dígito real: {rotulos[i].item()}")
    axes[i].axis('off')

plt.suptitle("Amostra dos Dados Reais - Dataset MNIST (Resolução 28x28)", fontsize=14, y=1.05)
plt.show()


### 4. **Treinamento**: Ciclo de treino com otimizador Adam e monitoramento de perda.

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

# Configuração do dispositivo (GPU se disponível)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = LeNet5(num_classes=10)
model.to(device)

# Hiperparâmetros
learning_rate = 0.001
batch_size = 64
num_epochs = 5

# Atualiza o DataLoader com um tamanho de lote maior para o treinamento
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Função de perda e otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Ciclo de treinamento
print(f'Treinando no dispositivo: {device}...')

# Coloca o modelo explicitamente no estado matemático de treino (importante para camadas como Dropout ou BatchNorm)
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Passagem direta (Forward)
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Retropropagação (Backward) e otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Soma a perda acumulada
        running_loss += loss.item()
        
        # Imprime o progresso a cada 100 iterações
        if (i+1) % 100 == 0:
            print(f'Época [{epoch+1}/{num_epochs}], Passo [{i+1}/{len(train_loader)}], Perda: {running_loss / 100:.4f}')
            running_loss = 0.0
            
print("\nTreinamento Finalizado!!")


### 5. **Avaliação**: Cálculo da precisão final em dados de teste.

In [ ]:
# O bloco de teste avaliará a precisão da IA usando um dataset totalmente novo que sobrou no set do MNIST.
# Isso é vital para comprovar que ela "aprendeu a lógica visual universal" dos números e não apenas "decorou as imagens do treino".

# 1. Carrega os 10.000 dados separados exclusivamente para Teste/Avaliação final. (Notar que train=False!)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) # Sem precisar dar shuffle aqui

print("Configurando modelo em modo de Avaliação...")
# Trava o comportamento do modelo em estado final de Averiguação (evita computos de dropouts/batch norms retroativos caso existissem)
model.eval() 

# Variáveis para contabilizarmos acertos!
correct = 0
total = 0

# Usamos a bandeira 'with torch.no_grad()' porque na fase de teste não queremos re-treinar a rede.
# Isso fala pro PyTorch não calcular nem rastrear nenhum Gradiente nessas operações.
# Consequência: Exige muuuuito menos memória e processa a velocidade da luz!
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # A rede inspeciona as 64 imagens do lote. Vai disparar os valores pros 10 últimos neurônios.
        # O neurônio que piscar com maior número na saída é a resposta que ela considerará o limite (exemplo: deduziu que é classe 4)
        outputs = model(images)
        
        # Extrai da avaliação a resposta máxima da matriz dela (A resposta que nós decidimos como voto final).
        _, predicted = torch.max(outputs.data, 1)
        
        # Contabilizando os resultados em massa do lote todo de uma vez!
        total += labels.size(0)  
        correct += (predicted == labels).sum().item() # Quantas vezes a resposta que prevemos bateu ser idêntica ao gabarito real.

accuracy = 100 * correct / total
print(f"\n====== RESULTADO DA INTELIGÊNCIA ARTIFICIAL ======")
print(f'Precisão final ponderada do nosso Modelo LeNet-5 avaliando mais de {total} Imagens inéditas de Teste: {accuracy:.2f}%')


### 6. **Salvamento do Modelo**: Exportação dos pesos treinados para um arquivo .pth.

In [ ]:
# O PyTorch trabalha exportando os chamados "state_dicts" (O dicionário que empacota todas as centenas de milhares de pesos alterados pela IA)
ARQUIVO_DESTINO = './minha_lenet5_mnist.pth'

# Literalmente gravando o objeto e salvando!
torch.save(model.state_dict(), ARQUIVO_DESTINO)

print(f"\nModelo salvo fisicamente com sucesso no formato .pth, sob o nome: '{ARQUIVO_DESTINO}'")
print("Você pode incorporar esse arquivo e rodar em aplicações Web, de Servidor...")

"""
Dica Bônus!
Na vida real, como seria pra você carregar a rede treinada dali de dentro pra realizar uma inferência nova? Simples assim:

# 1. Instanciar a casca puramente vazia 
modelo_carregado = LeNet5(num_classes=10)

# 2. Injetar a memória .pth externa pra dentro da RAM do respectivo PyTorch model
modelo_carregado.load_state_dict(torch.load('./minha_lenet5_mnist.pth'))

# 3. Declarar o modelo re-estabelecido em estado de Teste Inferencial (Pausando a engine interna que altera as features)
modelo_carregado.eval()
"""


### 7. **Análise de Erros**: Visualização das imagens que o modelo errou para entender suas limitações.

In [ ]:
import matplotlib.pyplot as plt

# Vamos criar uma célula visual dedicada apenas para 'caçar' aquelas poucas imagens que conseguiram enganar a rede neural

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
erro_qtd = 0

# Não atualizar pesos - Modo avaliador
model.eval()
with torch.no_grad():
    for images, labels in test_loader:  # varremos o set de Teste do zero
        images_device = images.to(device)
        labels_device = labels.to(device)
        
        outputs = model(images_device)
        _, predicted = torch.max(outputs.data, 1)
        
        # Criação de uma 'Máscara' identificadora Booleana (True/False) apontando os Lotes onde O palpite for DIVERGENTE do Rótulo Real
        erros_mask = predicted != labels_device
        
        # Agora extraímos cirurgicamente da lista APENAS as imagens falhas usando essa máscara, devolvendo-as pro PC pra podermos Printar nela
        imagens_fracasso = images[erros_mask.cpu()]  
        respostas_corretas = labels[erros_mask.cpu()]
        palpites_da_ia = predicted[erros_mask].cpu() 
        
        # Loop de plotagem, pegamos os limitados vacilos desse lote pra mostrar (limitando a 6 caixas)
        for j in range(len(respostas_corretas)):
            img = imagens_fracasso[j].squeeze().numpy()  # Desespremendo o tensor 3D de volta para pixels 2D do PyPlot
            
            axes[erro_qtd].imshow(img, cmap='gray')
            
            # Mostramos em Vermelho o Palpite Furado e o Valor Gabarito Real da caneta e nota
            axes[erro_qtd].set_title(f"IA disse: {palpites_da_ia[j].item()}\nEra de fato: {respostas_corretas[j].item()}", color='red')
            axes[erro_qtd].axis('off')
            
            erro_qtd += 1
            
            # Achou 6 deslizes totais? Tranca as varreduras, senão iríamos printar 150 imagens acidentais...
            if erro_qtd >= 6:
                break
        
        # Trava total iteradora 
        if erro_qtd >= 6:
            break

plt.suptitle("A Galeria do Fracasso: Imagens mal-escritas que derrubaram a LeNet-5", fontsize=14, y=1.05)
plt.show()


### 8. **Feature Maps**: Visualização das ativações internas das camadas convolucionais.

In [ ]:
import matplotlib.pyplot as plt

# ---- O CÉREBRO DA IA AGINDO ----
# Vamos ver como são os "Feature Maps" gerados pela 1ª camada Convolucional (Filtros extraindo peças da imagem)

# 1. Pega apenas 1 imagem aleatória de teste para usarmos de cobaia
iterador_teste = iter(test_loader)
imagens_teste, rotulos_teste = next(iterador_teste)

# O PyTorch trabalha com Lotes. Precisamos usar '.unsqueeze(0)' pra forjar um formato de [Lote, Canal, Altura, Largura] 
# Resultando num Tensor Falso composto de uma só imagem de tamanho [1, 1, 28, 28]
imagem_escolhida = imagens_teste[0].unsqueeze(0).to(device) 

model.eval()
with torch.no_grad():
    # 2. Ao invés da rede inteira (outputs = model(img)), chamamos EXCLUSIVAMENTE o tensor da camada 'conv1'
    mapas_de_caracteristica = model.conv1(imagem_escolhida)
    # Não esquecer do ReLU nela!
    mapas_de_caracteristica = F.relu(mapas_de_caracteristica)

print(f"Estrutura dimensional dos nossos Mapas Virtuais: {mapas_de_caracteristica.shape}")
print("Explicando as tuplas: (1 Imagem Lotezada | 6 Filtros Extratores Agindo | 28x28 Pixels graças ao Padding=2!)\n")

# 3. Plotagem Visual Lado a Lado
fig, axes = plt.subplots(1, 7, figsize=(18, 3)) # 1 Quadro Original + 6 Quadros Ativados do Conv1

# O Primeiro Quadrinho: Desfaz o Tensorzinho pra um numpy puramente cinza (Físico Real)
axes[0].imshow(imagens_teste[0].squeeze().numpy(), cmap='gray')
axes[0].set_title(f"Real Física (Nº {rotulos_teste[0].item()})")
axes[0].axis('off')

# Retorna à matriz da memória da Placa de Vídeo GPU -> Processador Central de volta pra gente ler com o PyPlot.
mapas_cpu = mapas_de_caracteristica.squeeze().cpu().numpy()

# Desenhando agora as 6 Versões Abstratas que o PyTorch enxerga nesse exato milissegundo
for i in range(6):
    # Visualmente veremos o que acendeu ou o que a convolução encontrou usando o efeito "viridis"
    axes[i+1].imshow(mapas_cpu[i], cmap='viridis')
    axes[i+1].set_title(f'Feature Map / Lente {i+1}')
    axes[i+1].axis('off')

plt.suptitle("A Magia da Extração: Como os 6 Filtros da Conv1 Abstraem as Bordas do Dígito Inédito!", fontsize=15, y=1.05)
plt.show()
